<a href="https://colab.research.google.com/github/neto995/data-science-tripleten/blob/main/megaline_classification(sprint_10_TripleTen).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mapa mental: Flujo de entrenamiento y evaluación

```text
users_behavior.csv
        ↓
1. Entender los datos
        ↓
2. Definir features y target
        ↓
3. Dividir los datos
   ├── TRAIN
   ├── VALIDATION
   └── TEST
        ↓
4. Entrenar varios modelos
   ├── Decision Tree
   ├── Random Forest
   └── Logistic Regression
        ↓
5. Ajustar hiperparámetros
        ↓
6. Comparar Accuracy en VALIDATION
        ↓
7. Elegir UN modelo
        ↓
8. Evaluarlo UNA sola vez en TEST
        ↓
9. Realizar prueba de cordura
        ↓
10. Conclusiones ejecutivas

## Diagrama: Train, Validation y Test

```text
100% DATASET
      │
      ├── TRAIN
      │      ↓
      │    fit()
      │      ↓
      │   aprende
      │
      ├── VALIDATION
      │      ↓
      │   predict()
      │      ↓
      │ escoger modelo
      │
      └── TEST
             ↓
          predict()
             ↓
       evaluación FINAL

# Paso 1: Comprensión de datos

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Proyecto 10/users_behavior.csv")

df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [ ]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [ ]:
df['is_ultra'].value_counts()

,count
is_ultra,
0,2229
1,985


In [ ]:
df['is_ultra'].value_counts(normalize=True)

,proportion
is_ultra,
0,0.693528
1,0.306472


###  Comprensión de datos: Definición de la prueba de cordura

No tengo valores nulos.

La variable objetivo `is_ultra` tiene la siguiente distribución:

- 2,229 Smart → **69.35%**
- 985 Ultra   → **30.64%**

Como **Smart es la clase mayoritaria**, una estrategia muy básica sería predecir que **Smart sea para todos los clientes**.

Sin aprender ningún patrón, este modelo tendría aproximadamente un **69.35% de accuracy**.

Por lo tanto:

> **69.35% es mi baseline o prueba de cordura.**

Si mi modelo supera ese **69.35%**, significa que está aprendiendo patrones útiles del comportamiento de los usuarios y no simplemente aprovechando que Smart es la clase mayoría.

Sin embargo, pero para este proyecto no basta con superar ligeramente el baseline. El requisito es alcanzar un **accuracy mínimo de 75%**.

```text
Baseline        → 69.35%
Proyecto mínimo → 75%
Modelo final    → queremos > 75%

# Paso 2: Definir features y target

In [ ]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

Uso todo el dataframe como **features**, excepto is_ultra, que es mi **target**.  
El target contiene la respuesta que queremos predecir:

* `0` → Smart
* `1` → Ultra

# Paso 3: Dividir datos

### Lógica de la División del dataset

**Train (60%)** → Uso el 60% del dataset para **crear el modelo y que se entrene** con los patrones que observa por sí mismo.

**Validation (20%)** → Le doy las **features tapándole las respuestas**. El modelo predice con `predict()` y comparo contra `target_valid`. Aquí elijo **qué modelo usar y sus hiperparámetros**.

**Test (20%)** → Una vez que ya escogí mi modelo y sus hiperparámetros, le doy las features del último 20%, que **nunca ha visto**, Predice y comparo contra `target_test`. Este es mi accuracy final (examen final).

```text
100% DATASET  
│  
├── 60%  TRAIN  
│      features + target  
│           ↓  
│         fit()  
│           ↓  
│        APRENDE  
│  
├── 20% VALIDATION  
│      features → predict()  
│           ↓  
│      vs. target_valid  
│           ↓  
│    ELIJO MODELO + HIPERPARÁMETROS  
│  
└── 20% TEST  
       features → predict()  
            ↓  
       vs. target_test  
            ↓  
       ACCURACY FINAL

### Competencia de modelos:

```text

                    60% TRAIN
          features_train + target_train
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
    Decision Tree  Random Forest  Logistic Reg.
          ↓            ↓            ↓
        fit()         fit()         fit()
          │            │            │
          └────────────┼────────────┘
                       ↓
                20% VALIDATION
                       ↓
              comparo Accuracy
                       ↓
               ¿QUIÉN GANÓ?

### Split

Primero: Spliteo, 40% spliteado por la mitad(validation, test) y el resto automáticamente es mi Train (60%).  
Segundo: Entreno mis tres modelos.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# separo 60% para entrenamiento y dejo 40% para validation + test almacenado en "rest"
features_train, features_rest, target_train, target_rest = train_test_split(
    features,
    target,
    test_size=0.4,
    random_state=54321
)

# Divido ese 40% restante en dos: 20% validation y 20% test
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest,
    target_rest,
    test_size=0.5,
    random_state=54321
)


# Paso 4: Competencia de modelos

Comparo los modelos usando sus **parámetros por defecto**, sin ajustar hiperparámetros.

## Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

#Paso 1: TRAIN MODEL

# 1.1 Creo el modelo
model = DecisionTreeClassifier(
    random_state=54321
)

#1.2. Entreno al modelo con el conjunto de entrenamiento
model.fit(features_train, target_train)


#Paso 2: VALIDATION

# 2.1 El modelo predice usando lo aprendido pero sin ver las respuestas
predictions_valid = model.predict(features_valid)

#2.2 Comparo su predicción hecha contra las respuestas reales
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("accuracy de la validación Decision Tree:", accuracy)

accuracy de la validación Decision Tree: 0.687402799377916


## Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier


#Paso 1: TRAIN MODEL

# 1.1 Creo el modelo
model = RandomForestClassifier(
    random_state=54321
)

#1.2. Entreno al modelo con el conjunto de entrenamiento
model.fit(features_train, target_train)


#Paso 2: VALIDATION

# 2.1 El modelo predice usando lo aprendido pero sin ver las respuestas
predictions_valid = model.predict(features_valid)

#2.2 Comparo su predicción hecha contra las respuestas reales
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("accuracy de la validación Random Forest:", accuracy)


accuracy de la validación Random Forest: 0.7822706065318819


## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

#Paso 1: TRAIN MODEL

# 1.1 Creo el modelo
model = LogisticRegression(
    random_state=54321
)

#1.2. Entreno al modelo con el conjunto de entrenamiento
model.fit(features_train, target_train)


#Paso 2: VALIDATION

# 2.1 El modelo predice usando lo aprendido pero sin ver las respuestas
predictions_valid = model.predict(features_valid)

#2.2 Comparo su predicción hecha contra las respuestas reales
accuracy = accuracy_score(
    target_valid,
    predictions_valid
)

print("accuracy de la validación Logistic Regression:", accuracy)

accuracy de la validación Logistic Regression: 0.7076205287713841


## Summary
resultados default sin hiperparámetros  


| Modelo                  | Accuracy de validación |
| ----------------------- | ---------------------: |
| **Decision Tree**       |                 0.6874 |
| **Random Forest**       |             **0.7823** |
| **Logistic Regression** |                 0.6750 |

**Mejor modelo:** `Random Forest` con un accuracy de validación de **78.23%** superando el baseline del 75%.


# Paso 5: Competencia de modelos con hiperparámetros

Comparo los modelos **ajustando sus hiperparámetros** para encontrar la mejor configuración de cada uno.


## Decision Tree Classifier

In [ ]:
best_accuracy = 0
best_depth = 0

for depth in range(1, 10):

    model = DecisionTreeClassifier(
        random_state=54321,
        max_depth=depth
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    accuracy = accuracy_score(
        target_valid,
        predictions_valid
    )

    print("max_depth:", depth, "accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_depth = depth

print("Mejor max_depth:", best_depth)
print("Mejor accuracy:", best_accuracy)

max_depth: 1 accuracy: 0.7216174183514774
max_depth: 2 accuracy: 0.7418351477449455
max_depth: 3 accuracy: 0.7651632970451011
max_depth: 4 accuracy: 0.744945567651633
max_depth: 5 accuracy: 0.7651632970451011
max_depth: 6 accuracy: 0.7542768273716952
max_depth: 7 accuracy: 0.7433903576982893
max_depth: 8 accuracy: 0.7511664074650077
max_depth: 9 accuracy: 0.7682737169517885
Mejor max_depth: 9
Mejor accuracy: 0.7682737169517885


## Random Forest Classifier

In [ ]:
best_accuracy = 0
best_est = 0
best_depth = 0

for est in range(10, 51, 10):
    for depth in range(1, 11):

        model = RandomForestClassifier(
            random_state=54321,
            n_estimators=est,
            max_depth=depth
        )

        model.fit(features_train, target_train)

        predictions_valid = model.predict(features_valid)

        accuracy = accuracy_score(
            target_valid,
            predictions_valid
        )

        print(
            "n_estimators:", est,
            "max_depth:", depth,
            "accuracy:", accuracy
        )

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_est = est
            best_depth = depth

print("Mejor n_estimators:", best_est)
print("Mejor max_depth:", best_depth)
print("Mejor accuracy:", best_accuracy)

n_estimators: 10 max_depth: 1 accuracy: 0.71850699844479
n_estimators: 10 max_depth: 2 accuracy: 0.7465007776049767
n_estimators: 10 max_depth: 3 accuracy: 0.7480559875583204
n_estimators: 10 max_depth: 4 accuracy: 0.7853810264385692
n_estimators: 10 max_depth: 5 accuracy: 0.7807153965785381
n_estimators: 10 max_depth: 6 accuracy: 0.7838258164852255
n_estimators: 10 max_depth: 7 accuracy: 0.7884914463452566
n_estimators: 10 max_depth: 8 accuracy: 0.7993779160186625
n_estimators: 10 max_depth: 9 accuracy: 0.7869362363919129
n_estimators: 10 max_depth: 10 accuracy: 0.80248833592535
n_estimators: 20 max_depth: 1 accuracy: 0.702954898911353
n_estimators: 20 max_depth: 2 accuracy: 0.7293934681181959
n_estimators: 20 max_depth: 3 accuracy: 0.7713841368584758
n_estimators: 20 max_depth: 4 accuracy: 0.7807153965785381
n_estimators: 20 max_depth: 5 accuracy: 0.7791601866251944
n_estimators: 20 max_depth: 6 accuracy: 0.7791601866251944
n_estimators: 20 max_depth: 7 accuracy: 0.7822706065318819
n

## Logistic Regression

C controla la regularización del modelo.

In [ ]:
best_accuracy = 0
best_c = 0

for c in [0.01, 0.1, 1, 10, 100]:

    model = LogisticRegression(
        random_state=54321,
        solver='liblinear',
        C=c
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    accuracy = accuracy_score(
        target_valid,
        predictions_valid
    )

    print("C:", c, "accuracy:", accuracy)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_c = c

print("Mejor C:", best_c)
print("Mejor accuracy:", best_accuracy)

C: 0.01 accuracy: 0.6734059097978227
C: 0.1 accuracy: 0.6780715396578538
C: 1 accuracy: 0.6780715396578538
C: 10 accuracy: 0.6780715396578538
C: 100 accuracy: 0.6780715396578538
Mejor C: 0.1
Mejor accuracy: 0.6780715396578538


todos los valores probados obtuvieron prácticamente el mismo accuracy, así que meter hiperparámetros no ayuda.

# Paso 6: comparar Accuracy en VALIDATION

## Competencia de modelos con hiperparámetros.

| Modelo | Accuracy Default | Mejor configuración | Accuracy con hiperparámetros |
|---|---:|---|---:|
| Decision Tree | 0.6874 | `max_depth=10` | **0.7823** |
| Random Forest | **0.7823** | `n_estimators=10`, `max_depth=10` | **0.8025** 🏆 |
| Logistic Regression | 0.6750 | `C=0.01`* | **0.6781** |



# Paso 7: Elergir UN modelo

### Summary

- **Decision Tree:** mejoró de `68.74%` → `78.23%`.
- **Random Forest:** mejoró de `78.23%` → `80.25%` y obtuvo el mejor resultado. 🏆
- **Logistic Regression:** se mantuvo alrededor de `67.8%`.
- **Modelo ganador en Validation:** `RandomForestClassifier`.
- **Configuración ganadora:** `n_estimators=10`, `max_depth=10`.
- **Siguiente paso:** evaluar el modelo ganador con el conjunto de **Test**.

> *En Logistic Regression los valores de `C = 0.01, 0.1, 1, 10, 100` dieron prácticamente el mismo accuracy. `0.01` aparece como el mejor porque fue el primero en alcanzar ese resultado.*

# Paso 8: Evaluarlo en TEST

In [ ]:
# Paso 3: TEST

# 3.1 Creo el modelo ganador con los mejores hiperparámetros
final_model = RandomForestClassifier(
    random_state=54321,
    n_estimators=10,
    max_depth=10
)

# 3.2 Entreno el modelo
final_model.fit(features_train, target_train)

# 3.3 El modelo hace el examen final con features que nunca usamos
predictions_test = final_model.predict(features_test)

# 3.4 Comparo sus predicciones contra las respuestas reales
accuracy_test = accuracy_score(
    target_test,
    predictions_test
)

print("Accuracy final en Test:", accuracy_test)

Accuracy final en Test: 0.8242612752721618


### Test Conclusion

El modelo seleccionado fue `RandomForestClassifier` con:

- `n_estimators = 10`
- `max_depth = 10`
- `random_state = 54321`

El modelo obtuvo un **accuracy de 82.43% en el conjunto de Test**, superando:

- **Baseline:** 69.35%
- **Accuracy mínimo requerido:** 75%
- **Validation accuracy:** 80.25%

Significa que de los clientes que el modelo nunca utilizó ni para entrenarse ni para hiperparámetros clasifica 82% de las veces correctamente.

**Final Test Accuracy: 82.43%**

# Paso 9: Prueba de cordura

¿Mi modelo realmente aprendió a identificar el perfil de cada cliente, o simplemente se está aprovechando de que la mayoría prefiere el plan **Smart** y entonces predice que un cliente será Smart solo por ser mayoría?

Si mi modelo me diera un resultado menor a mi prueba de cordura lo lógico sería usar el 70% como piso, y ofrecerle a todos los clientes el Plan Smart así al menos le atino al 70% de las veces.

In [ ]:
from sklearn.dummy import DummyClassifier

# Creo un modelo "tonto" que siempre predice la clase más frecuente
dummy_model = DummyClassifier(strategy='most_frequent')

# Aprende cuál es la clase más frecuente en TRAIN
dummy_model.fit(features_train, target_train)

# Predice siempre esa clase en TEST
dummy_predictions = dummy_model.predict(features_test)

# Calculo su accuracy
dummy_accuracy = accuracy_score(
    target_test,
    dummy_predictions
)

print("Baseline accuracy:", dummy_accuracy)
print("Random Forest accuracy:", accuracy_test)

Baseline accuracy: 0.7247278382581649
Random Forest accuracy: 0.8242612752721618


# Paso 10: Conclusiones

## Conclusiones

Nuestro modelo predictivo sí es viable ya que supera todas las referencias de comparación:

1. **Baseline inicial:** la clase mayoritaria representa aproximadamente el 69.35% del dataset, por lo que predecir siempre Smart sería una estrategia simple pero limitada.
2. **Prueba de cordura con `DummyClassifier`:** en el conjunto de Test, predecir siempre la clase mayoritaria obtiene un accuracy de **72.47%**.
3. **Umbral mínimo del proyecto:** el modelo debía superar un accuracy de **75%**.
4. **Random Forest sin ajuste:** el modelo base ya mostraba un desempeño competitivo, pero la búsqueda de hiperparámetros permitió mejorar su rendimiento.
5. **Modelo final:** `RandomForestClassifier` con `n_estimators=10` y `max_depth=10` obtuvo un accuracy final de **82.43%** en Test.

El modelo final supera al `DummyClassifier` por aproximadamente **10 puntos porcentuales**, lo que indica que no se limita a predecir Smart por ser la clase mayoritaria.

En cambio, está utilizando patrones presentes en variables como llamadas, minutos, mensajes y consumo de internet para clasificar de forma más efectiva a los clientes entre los planes Smart y Ultra.

#Convertir modelo a Java Script
Sigue convertir el modelo an onnx es un convertidor de lenguajes en este caso paso python a java script.

In [ ]:
! pip install skl2onnx onnx

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [
    ("input", FloatTensorType([None, 4]))
]

onnx_model = convert_sklearn(
    final_model,
    initial_types=initial_type,
    options={
        id(final_model): {
            "zipmap": False
        }
    }
)

with open("megaline_random_forest.onnx", "wb") as file:
    file.write(onnx_model.SerializeToString())

### Descargo ahora el modelo en versión .onnx.

In [81]:
#from google.colab import files

#files.download("megaline_random_forest.onnx")

In [85]:
test_cases = pd.DataFrame([
    {
        "calls": 0,
        "minutes": 0,
        "messages": 0,
        "mb_used": 0
    },
    {
        "calls": 40,
        "minutes": 130,
        "messages": 80,
        "mb_used": 20000
    },
    {
        "calls": 250,
        "minutes": 1500,
        "messages": 250,
        "mb_used": 50000
    }
])

print(final_model.predict(test_cases))
print(final_model.predict_proba(test_cases))

[1 0 1]
[[0.28333333 0.71666667]
 [0.55086894 0.44913106]
 [0.1        0.9       ]]
